In [5]:
from minedatabase.pickaxe import Pickaxe
from ergochemics.standardize import standardize_smiles
from functools import lru_cache
from hydra import initialize, compose
import polars as pl
from pathlib import Path

with initialize(version_base=None, config_path="../configs/filepaths"):
    cfg = compose(config_name="filepaths")

@lru_cache(maxsize=10000)
def std_smi(smi: str) -> str:
    return standardize_smiles(smi, neutralization_method="simple")

def std_rxn(rxn: str) -> str:
    lhs, rhs = [side.split(".") for side in rxn.split(">>")]
    lhs = sorted([std_smi(smi) for smi in lhs])
    rhs = sorted([std_smi(smi) for smi in rhs])
    return ".".join(lhs) + ">>" + ".".join(rhs)

In [11]:
cutoff_date = 2015
pub = pl.read_parquet(Path(cfg.raw_data) / "rxn_pub_dates.parquet")
pub = pub.explode("publication_dates")
pub = pub.group_by("id").agg(pl.col("publication_dates").min()).filter(pl.col("publication_dates") >= cutoff_date).rename({"publication_dates": "pub_date"})
pub.head()

id,pub_date
str,i32
"""f73b53d402c070179b8be59d2f035d…",2020
"""281ce48dda6c293f3d7818258edc62…",2016
"""9555b922e67a1c59a8ea162347c51a…",2019
"""0a3434def8c8206409e3d65436dc24…",2019
"""81e3cb717a371e70321fb142e39ef9…",2017


In [12]:
exp = Pickaxe()
exp.load_pickled_pickaxe(Path(cfg.expansions) / "1_steps_after_2015_cpds_rules_mechinferred_dt_069_rules_before_2015_w_coreactants_aplusb_True.pk")

----------------------------------------
Intializing pickaxe object

Done intializing pickaxe object
----------------------------------------

Loading /home/stef/quest_data/cgr/expansions/1_steps_after_2015_cpds_rules_mechinferred_dt_069_rules_before_2015_w_coreactants_aplusb_True.pk pickled data.
Loaded 122189 compounds
Loaded 161821 reactions
Loaded 11612 operators
Loaded 581 coreactants
Loaded 1 generation
Took 3.3170435428619385


[14:44:04] WARNING: not removing hydrogen atom without neighbors


In [14]:

exp.reactions

{'R05abb3934c64706800124e64fa6de255b3328caafde3d323fe4d024d747f63f7': {'_id': 'R05abb3934c64706800124e64fa6de255b3328caafde3d323fe4d024d747f63f7',
  'Reactants': [(1, 'C1948d02a7a8b82013585a8da68cac68f74de32df'),
   (1, 'C488c3ce5e06c8373bf0c7459f6411a29444400be')],
  'Products': [(1, 'X662c3c548e200875cdfe6560db6a67842229ae64'),
   (1, 'Cfb7ff73b60c539c1f47108de9f162ae483dd09ae')],
  'Operators': {'1939_1', '1939_1_0', '1939_1_0_0'},
  'SMILES_rxn': '(1) O=C(O)C=C(O)C(=O)O + (1) CC(C(=O)SCCNC(=O)CCNC(=O)C(O)C(C)(C)COP(=O)(O)OP(=O)(O)OCC1OC(n2cnc3c(N)ncnc32)C(O)C1OP(=O)(O)O)C1(O)CCC2C3CCC4=CC(=O)CCC4(C)C3CCC21C => (1) CC(C(=O)OC(=O)C(O)=CC(=O)O)C1(O)CCC2C3CCC4=CC(=O)CCC4(C)C3CCC21C + (1) CC(C)(COP(=O)(O)OP(=O)(O)OCC1OC(n2cnc3c(N)ncnc32)C(O)C1OP(=O)(O)O)C(O)C(=O)NCCC(=O)NCCS',
  'am_rxn': '[O:1]=[C:2]([OH:3])[CH:4]=[C:5]([OH:6])[C:7](=[O:8])[OH:9].[CH3:10][CH:11]([C:12](=[O:13])[S:14][CH2:15][CH2:16][NH:17][C:18](=[O:19])[CH2:20][CH2:21][NH:22][C:23](=[O:24])[CH:25]([OH:26])[C:27]([CH3: